In [ ]:
# Block 1: main config only
mode = "solo"  # "solo" or "agents"
goal = """Điền goal chính ở đây."""
task = """Điền task cụ thể ở đây."""
role = "SOLO"
roles = ["DEV", "REVIEW"]
start_role = ""
team = False
no_parallel = False
max_turns = 50
timeout_s = 3000


In [ ]:
# Block 2: main runner + reusable audit helpers
from pprint import pprint
import agents

settings = agents.load_simple_toml("config.toml")
settings["prompts_dir"] = "prompts"

def effective_goal():
    parts = [str(x).strip() for x in [goal, task] if str(x or "").strip()]
    return "\n\n".join(parts)

def role_snapshot(target_role):
    return agents.http_json("GET", f"/api/admin/role/{target_role}")

def server_routes():
    return agents.http_json("GET", "/api/admin/routes")

def send_browser_command(target_role, action, payload=None, timeout=120):
    return agents.run_command(target_role, action, payload or {}, timeout=timeout, print_every=1.0)

def dump_buttons(target_role, limit=80):
    result = send_browser_command(target_role, "DUMP_BUTTONS", {"limit": limit}, timeout=120)
    pprint(result)
    return result

def recent_messages(target_role, n=5):
    snap = role_snapshot(target_role)
    messages = ((snap.get("dom_info") or {}).get("messages") or {}).get("messages") or []
    recent = messages[-n:]
    for i, item in enumerate(recent, start=max(1, len(messages) - len(recent) + 1)):
        role_name = item.get("role", "?")
        text = str(item.get("text") or "").replace("\n", " ")
        print(f"{i:03d} [{role_name}] {text[:500]}")
    return recent

def recent_events(target_role="", limit=30):
    path = f"/api/admin/events?limit={limit}"
    if target_role:
        path += f"&role={target_role}"
    data = agents.http_json("GET", path)
    for item in data.get("events", []):
        print(f"{item.get('ts')} [{item.get('role')}] {item.get('event')} {item.get('command_id', '')}")
    return data

def run_current_config():
    text = effective_goal()
    if not text:
        raise ValueError("Fill goal/task in Block 1 first.")
    agents.load_agent_core()
    core = agents.__dict__
    if mode == "solo":
        selected = [str(role).upper().strip() or "SOLO"]
        agents.ACTIVE_ROLES = selected
        agents.log_roles_status(selected)
        return agents.run_agent_loop(selected, text, start_role=selected[0], max_turns=max_turns, timeout_s=timeout_s, core=core, settings=settings)
    if mode == "agents":
        selected = agents.normalize_role_list(roles)
        if team:
            selected = agents.resolve_team_roles(selected, agents.discover_prompt_roles("prompts"))
        agents.ACTIVE_ROLES = selected
        agents.log_roles_status(selected)
        if team:
            return agents.run_team_loop(selected, text, start_role=start_role, max_turns=max_turns, timeout_s=timeout_s, no_parallel=no_parallel, core=core, settings=settings)
        return agents.run_agent_loop(selected, text, start_role=start_role, max_turns=max_turns, timeout_s=timeout_s, core=core, settings=settings)
    raise ValueError(f"Unsupported mode: {mode!r}")

result = run_current_config()
result


In [ ]:
# Block 3: audit visible/clickable buttons for one role
button_role = role if mode == "solo" else (roles[0] if roles else "A")
dump_buttons(button_role)


In [ ]:
# Block 4: audit recent parsed messages for one role
message_role = role if mode == "solo" else (roles[0] if roles else "A")
recent_messages(message_role, n=5)


In [ ]:
# Block 5: audit server-side snapshot for one role
snapshot_role = role if mode == "solo" else (roles[0] if roles else "A")
pprint(role_snapshot(snapshot_role))


In [ ]:
# Block 6: audit backend routes and samples
pprint(server_routes())


In [ ]:
# Block 7: audit recent backend events
recent_events(limit=30)
